In [1]:
import hereutil
hereutil.add_to_sys_path(hereutil.here())
from src.common_basis_gizmosql import *

In [2]:
p(f('all_places_of_publication').filter(c('source').is_in(['vd16','vd17','vd18'])).group_by(c('place_type'), c('place_of_publication')).agg(nw.len()).sort('len', descending=True)).to_pandas()

,place_type,place_of_publication,len
0,standardized,Leipzig,121300
1,unstandardized,[S.l.],92437
2,unstandardized,Leipzig,75037
3,standardized,Wittenberg,51976
4,standardized,Jena,46039
...,...,...,...
19956,unstandardized,Mongvntiae,1
19957,unstandardized,Hala Saxonum,1
19958,unstandardized,Dasburgi,1
19959,unstandardized,Soteropoli Anhalt.,1


In [9]:
d = p(
    f('all_places_of_publication').filter(c('source').is_in(['vd16','vd17','vd18']), c('place_type')=='unstandardized').rename({'place_of_publication': 'place_of_publication_unstandardized'})
    .join(
        f('all_places_of_publication').filter(c('source').is_in(['vd16','vd17','vd18']), c('place_type')=='standardized'),
        left_on=['source', 'record_number'],
        right_on=['source', 'record_number'],
        how='full'
    )
    .select(nw.coalesce(c('source'), c('source_right')), nw.coalesce(c('record_number'), c('record_number_right')), nw.coalesce(c('place_of_publication'), c('place_of_publication_unstandardized')).alias('place_of_publication'))
    .with_columns(place_of_publication=c('place_of_publication').str.replace_all('Frankfurt.*Main.*', 'Frankfurt am Main').str.replace_all('Frankfurt.*Oder.*', 'Frankfurt an der Oder'))
    .group_by(c('source'), c('place_of_publication'))
    .agg(nw.len())
    .with_columns(prop=c('len')/c('len').sum().over('source'))
    .join(f('geonames').filter(c('feature_class') == 'P').filter(c('population')==c('population').max().over('name')).select(c('name'), c('latitude'), c('longitude')), how='left', left_on='place_of_publication', right_on='name')
).to_pandas()

In [76]:
import folium
import pandas as pd
m = folium.Map(location=[48, 11], zoom_start=5, tiles='CartoDB.Positron')
colors = {'vd16': 'blue', 'vd17': 'green', 'vd18': 'red' }
for _, row in d.iterrows():
    if pd.notnull(row['latitude']) and pd.notnull(row['longitude']):
        folium.CircleMarker(location=[row['latitude'], row['longitude']], color=colors[row['source']] if row['source'] in colors else 'gray', radius=300*row['prop'], popup=f"{row['place_of_publication']}: {row['len']}").add_to(m)
m.save("map.html")

In [ ]:
d = p(
    f('all_places_of_publication').filter(c('place_type')=='unstandardized').rename({'place_of_publication': 'place_of_publication_unstandardized'})
    .join(
        f('all_places_of_publication').filter(c('place_type')=='standardized'),
        left_on=['source', 'record_number'],
        right_on=['source', 'record_number'],
        how='full'
    )
    .select(nw.coalesce(c('source'), c('source_right')), nw.coalesce(c('record_number'), c('record_number_right')), nw.coalesce(c('place_of_publication'), c('place_of_publication_unstandardized')).alias('place_of_publication'))
    .with_columns(place_of_publication=c('place_of_publication').str.replace_all('Frankfurt.*Main.*', 'Frankfurt am Main').str.replace_all('Frankfurt.*Oder.*', 'Frankfurt an der Oder'))
    .join(f('all_years_of_publication'), on="record_number")
    .with_columns(century_of_publication=(c('year_of_publication')/100).floor()*100)
    .group_by(c('century_of_publication'), c('place_of_publication'))
    .agg(nw.len())
    .with_columns(prop=c('len')/c('len').sum().over('century_of_publication'))
    .join(f('geonames').filter(c('feature_class') == 'P').filter(c('population')==c('population').max().over('name')).select(c('name'), c('latitude'), c('longitude')), how='left', left_on='place_of_publication', right_on='name')
    
).to_pandas()

In [14]:
import gspread
from hereutil import here
gc = gspread.oauth(credentials_filename=here('gsheets_credentials.json'), authorized_user_filename=here('gsheets_authorized_user.json'))

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=596097845664-00e1b95p3ht8t5f5co5g8lcg6t14l4un.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A57355%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fspreadsheets+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&state=aK5GAZ6dZGHv9IsY4bkzxB2eYdQ4Eo&code_challenge=0Bh5RAdzaPIhkbBVGJ5aB3Kbm_872x3EWdXfGsrhM7s&code_challenge_method=S256&access_type=offline


In [21]:
place_coords = p(f('all_places_of_publication').filter(c('source').is_in(['vd16','vd17','vd18']), c('place_type')=='unstandardized').rename({'place_of_publication': 'place_of_publication_unstandardized'})
    .join(
        f('all_places_of_publication').filter(c('source').is_in(['vd16','vd17','vd18']), c('place_type')=='standardized'),
        left_on=['source', 'record_number'],
        right_on=['source', 'record_number'],
        how='full'
    )
    .select(nw.coalesce(c('source'), c('source_right')), nw.coalesce(c('record_number'), c('record_number_right')), nw.coalesce(c('place_of_publication'), c('place_of_publication_unstandardized')).alias('place_of_publication'))
    .with_columns(place_of_publication=c('place_of_publication').str.replace_all('Frankfurt.*Main.*', 'Frankfurt am Main').str.replace_all('Frankfurt.*Oder.*', 'Frankfurt an der Oder'))
    .join(f('all_years_of_publication'), on="record_number")
    .with_columns(century_of_publication=(c('year_of_publication')/100).floor()*100)
    .group_by(c('century_of_publication'), c('place_of_publication'))
    .agg(nw.len())
    .with_columns(prop=c('len')/c('len').sum().over('century_of_publication'))
    .join(f('geonames').filter(c('feature_class') == 'P').filter(c('population')==c('population').max().over('name')).select(c('name'), c('latitude'), c('longitude')), how='inner', left_on='place_of_publication', right_on='name')
    .select(c('place_of_publication'), c('latitude'), c('longitude'))
    .unique()
).to_pandas()
place_coords

,place_of_publication,latitude,longitude
0,Ingolstadt,48.76508,11.42372
1,Halle (Saale),51.48158,11.97947
2,Ronneburg,50.86340,12.18666
3,Würzburg,49.79391,9.95121
4,Marburg,-27.56609,152.59705
...,...,...,...
1580,Chalons,45.72564,-0.96270
1581,Hardenberg,52.57583,6.61944
1582,Halla,-15.15278,-69.41884
1583,Chalon-sur-Saône,46.78112,4.85372


In [22]:
gc.open_by_key("1kv7RLk62bUJCb-je94ajlEsIsoIBXweZChLBnAIxF-E").worksheet("place_coordinates").update([place_coords.columns.values.tolist()] + place_coords.values.tolist())

{'spreadsheetId': '1kv7RLk62bUJCb-je94ajlEsIsoIBXweZChLBnAIxF-E',
 'updatedRange': 'place_coordinates!A1:C1586',
 'updatedRows': 1586,
 'updatedColumns': 3,
 'updatedCells': 4758}

In [7]:
(f('geonames').head(10)).collect().to_pandas()

,geonameid,name,asciiname,latitude,longitude,feature_class,feature_code,country_code,cc2,admin1_code,admin2_code,admin3_code,admin4_code,population,elevation,dem,timezone,modification_date
0,2994701,Roc Meler,Roc Meler,42.58765,1.74180,T,PK,AD,"AD,FR",02,NaN,NaN,NaN,0,2811.0,2348,Europe/Andorra,2023-10-03
1,3017832,Pic de les Abelletes,Pic de les Abelletes,42.52535,1.73343,T,PK,AD,FR,A9,66,663,66146,0,NaN,2411,Europe/Andorra,2014-11-05
2,3017833,Estany de les Abelletes,Estany de les Abelletes,42.52915,1.73362,H,LK,AD,FR,A9,NaN,NaN,NaN,0,NaN,2260,Europe/Andorra,2014-11-05
3,3023203,Port Vieux de la Coume d’Ose,Port Vieux de la Coume d'Ose,42.62568,1.61823,T,PASS,AD,NaN,00,NaN,NaN,NaN,0,NaN,2687,Europe/Andorra,2014-11-05
4,3029315,Port de la Cabanette,Port de la Cabanette,42.60000,1.73333,T,PASS,AD,"AD,FR",B3,09,091,09139,0,NaN,2379,Europe/Andorra,2014-11-05
5,3034945,Roc de Port Dret,Roc de Port Dret,42.60288,1.45736,T,PK,AD,"AD,FR",04,NaN,NaN,NaN,0,2735.0,2650,Europe/Andorra,2023-12-24
6,3038814,Costa de Xurius,Costa de Xurius,42.50692,1.47569,T,SLP,AD,NaN,07,NaN,NaN,NaN,0,NaN,1839,Europe/Andorra,2015-03-08
7,3038815,Font de la Xona,Font de la Xona,42.55003,1.44986,H,SPNG,AD,NaN,04,NaN,NaN,NaN,0,NaN,1976,Europe/Andorra,2010-01-11
8,3038816,Xixerella,Xixerella,42.55327,1.48736,P,PPL,AD,NaN,04,NaN,NaN,NaN,0,NaN,1417,Europe/Andorra,2009-04-24
9,3038818,Riu Xic,Riu Xic,42.57165,1.67554,H,STM,AD,NaN,02,NaN,NaN,NaN,0,NaN,1851,Europe/Andorra,2014-12-03


In [1]:
import hereutil
hereutil.add_to_sys_path(hereutil.here())
from src.common_basis_s3 import *

  0%|          | 0/72 [00:00<?, ?it/s]

In [5]:
bnf.head(5).collect(backend='polars').to_native()

record_number,field_number,subfield_number,field_code,subfield_code,value
i32,i32,i32,str,str,str
38,2,1,"""000""","""""",""" n0 d922 45a """
493,2,1,"""000""","""""",""" n0 d922 45a """
656,2,1,"""000""","""""",""" n0 d922 45a """
861,2,1,"""000""","""""",""" n0 d922 45a """
1009,2,1,"""000""","""""",""" n0 d922 45a """


In [2]:
import hereutil
hereutil.add_to_sys_path(hereutil.here())
from src.common_basis_local import *

  0%|          | 0/74 [00:00<?, ?it/s]

In [3]:
bnf.head(5).collect(backend='polars').to_native()

record_number,field_number,subfield_number,field_code,subfield_code,value
i32,i32,i32,str,str,str
38,2,1,"""000""","""""",""" n0 d922 45a """
493,2,1,"""000""","""""",""" n0 d922 45a """
656,2,1,"""000""","""""",""" n0 d922 45a """
861,2,1,"""000""","""""",""" n0 d922 45a """
1009,2,1,"""000""","""""",""" n0 d922 45a """
